In [3]:
import time
import json
import requests

from collections import deque
from urllib.parse import urljoin, urlparse, urldefrag

from bs4 import BeautifulSoup

# ============================================
# SELENIUM
# ============================================

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By

from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# ============================================
# REPORTLAB
# ============================================

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak
)

from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

# ============================================
# SETTINGS
# ============================================

MAX_PAGES = 50

visited = set()

# ============================================
# CHROME DRIVER
# ============================================

chrome_options = Options()

chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

# ============================================
# AI SUMMARY
# ============================================

def generate_ai_summary(text):

    try:

        prompt = f"""
        Analyze this webpage professionally.

        Give:
        1. Main Purpose
        2. Important Information
        3. Products/Services Mentioned
        4. Technologies Mentioned
        5. Business Value
        6. Final Summary

        CONTENT:
        {text[:5000]}
        """

        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3",
                "prompt": prompt,
                "stream": False
            },
            timeout=180
        )

        return response.json()["response"]

    except Exception as e:

        return f"AI Summary Failed: {e}"

# ============================================
# SCROLL PAGE
# ============================================

def scroll_page():

    last_height = driver.execute_script(
        "return document.body.scrollHeight"
    )

    for _ in range(5):

        driver.execute_script(
            "window.scrollTo(0, document.body.scrollHeight);"
        )

        time.sleep(2)

        new_height = driver.execute_script(
            "return document.body.scrollHeight"
        )

        if new_height == last_height:
            break

        last_height = new_height

# ============================================
# EXTRACT CONTENT
# ============================================

def scrape_page(url):

    try:

        print(f"\n[SCRAPING] {url}")

        driver.get(url)

        time.sleep(3)

        scroll_page()

        html = driver.page_source

        soup = BeautifulSoup(html, "lxml")

        # remove junk
        for tag in soup(["script", "style", "noscript"]):
            tag.extract()

        # TITLE
        title = soup.title.string.strip() if soup.title else "No Title"

        # HEADINGS
        headings = []

        for tag in soup.find_all(["h1", "h2", "h3"]):

            text = tag.get_text(" ", strip=True)

            if text:
                headings.append(text)

        # PARAGRAPHS
        paragraphs = []

        for p in soup.find_all(["p", "li"]):

            text = p.get_text(" ", strip=True)

            if len(text) > 40:
                paragraphs.append(text)

        # BUTTON TEXT
        buttons = []

        for b in soup.find_all(["button", "a"]):

            text = b.get_text(" ", strip=True)

            if 2 < len(text) < 80:
                buttons.append(text)

        # FULL TEXT
        content = soup.get_text(" ", strip=True)

        content = " ".join(content.split())

        # AI SUMMARY
        ai_summary = generate_ai_summary(content)

        data = {
            "url": url,
            "title": title,
            "headings": headings[:50],
            "paragraphs": paragraphs[:50],
            "buttons": buttons[:50],
            "content": content[:15000],
            "ai_summary": ai_summary
        }

        return data

    except Exception as e:

        print(f"[ERROR] {e}")

        return None

# ============================================
# FIND INTERNAL LINKS
# ============================================

def extract_internal_links(url):

    links = set()

    try:

        soup = BeautifulSoup(driver.page_source, "lxml")

        domain = urlparse(url).netloc

        for tag in soup.find_all("a", href=True):

            href = tag["href"]

            full_url = urljoin(url, href)

            # REMOVE #fragment
            full_url = urldefrag(full_url)[0]

            parsed = urlparse(full_url)

            # ONLY SAME DOMAIN
            if parsed.netloc == domain:

                clean = (
                    parsed.scheme +
                    "://" +
                    parsed.netloc +
                    parsed.path
                )

                if clean not in visited:
                    links.add(clean)

    except Exception as e:

        print(e)

    return links

# ============================================
# CRAWLER
# ============================================

def crawl_website(start_url):

    queue = deque([start_url])

    all_data = []

    while queue and len(visited) < MAX_PAGES:

        current_url = queue.popleft()

        if current_url in visited:
            continue

        visited.add(current_url)

        print("\n================================")
        print(f"[VISITING] {current_url}")
        print("================================")

        # SCRAPE PAGE
        data = scrape_page(current_url)

        if data:
            all_data.append(data)

        # FIND NEW LINKS
        new_links = extract_internal_links(current_url)

        for link in new_links:

            if link not in visited:
                queue.append(link)

    return all_data

# ============================================
# SAVE JSON
# ============================================

def save_json(data):

    with open(
        "complete_website_data.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )

    print("\n[SAVED] complete_website_data.json")

# ============================================
# GENERATE PDF
# ============================================

def generate_pdf(data):

    pdf = SimpleDocTemplate(
        "complete_website_report.pdf",
        pagesize=letter
    )

    styles = getSampleStyleSheet()

    elements = []

    title = Paragraph(
        "<b>COMPLETE WEBSITE AI ANALYSIS REPORT</b>",
        styles["Title"]
    )

    elements.append(title)

    elements.append(Spacer(1, 20))

    for i, page in enumerate(data):

        elements.append(
            Paragraph(
                f"<b>Page {i+1}</b>",
                styles["Heading1"]
            )
        )

        elements.append(Spacer(1, 10))

        elements.append(
            Paragraph(
                f"<b>URL:</b> {page['url']}",
                styles["BodyText"]
            )
        )

        elements.append(Spacer(1, 8))

        elements.append(
            Paragraph(
                f"<b>Title:</b> {page['title']}",
                styles["BodyText"]
            )
        )

        elements.append(Spacer(1, 10))

        headings = "<br/>".join(page["headings"])

        elements.append(
            Paragraph(
                f"<b>Headings:</b><br/>{headings}",
                styles["BodyText"]
            )
        )

        elements.append(Spacer(1, 10))

        elements.append(
            Paragraph(
                f"<b>AI Summary:</b><br/>{page['ai_summary']}",
                styles["BodyText"]
            )
        )

        elements.append(Spacer(1, 20))

        preview = page["content"][:4000]

        elements.append(
            Paragraph(
                f"<b>Content Preview:</b><br/>{preview}",
                styles["BodyText"]
            )
        )

        elements.append(PageBreak())

    pdf.build(elements)

    print("\n[SAVED] complete_website_report.pdf")

# ============================================
# MAIN
# ============================================

if __name__ == "__main__":

    start_url = input("Enter Website URL: ").strip()

    if not start_url.startswith("http"):
        start_url = "https://" + start_url

    start = time.time()

    data = crawl_website(start_url)

    save_json(data)

    generate_pdf(data)

    end = time.time()

    print("\n================================")
    print("CRAWLING COMPLETED")
    print("================================")

    print(f"\nPages Crawled: {len(data)}")

    print(f"Time Taken: {round(end-start,2)} sec")

    driver.quit()

Enter Website URL: https://aicrops.cloud/

[VISITING] https://aicrops.cloud/

[SCRAPING] https://aicrops.cloud/

[SAVED] complete_website_data.json

[SAVED] complete_website_report.pdf

CRAWLING COMPLETED

Pages Crawled: 1
Time Taken: 190.38 sec


In [4]:
import os

print(os.path.abspath("complete_website_report.pdf"))

C:\Users\Asus\complete_website_report.pdf
